# Figure 4 and Tables 1–2: one-dimensional method comparison

Run all cells to generate the figure and tables as PNG and CSV files in `results/`.
The experiment uses 1,000 unknowns, 50 DCT measurements, and 150 iterations.
Flexible methods can stop earlier at their breakdown tolerance.

The IAS FLSQR-W row in Table 2 uses the MM FLSQR-W result.
Figure 4 uses the IAS FLSQR-W history.


In [ ]:
print("Device: CPU")

from pathlib import Path
import os, sys
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
# Use eight BLAS threads for the dense inverse and projected solves.
os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"
from threadpoolctl import threadpool_limits

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiment_1d import make_problem, set_style, summarize, logged_solve
thread_limits = threadpool_limits(limits=8)
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
set_style()
A, L, Linv, Lpinv_factory, b, ground_truth, noise_var = make_problem()
# Use a dense inverse to keep the floating-point summation order fixed.
# Flexible methods near breakdown are sensitive enough that cumulative sums
# can change the final projected condition number at the paper's precision.
import operators
import scipy.sparse as sps
Linv = operators.MatrixLinearOperator(np.linalg.inv(L.A.toarray()))
def Lpinv_factory(wr):
    return Linv @ operators.MatrixLinearOperator(sps.diags(1.0 / wr))
n = len(ground_truth)
print(f"{n} unknowns, {len(b)} DCT measurements; relative noise 0.03; seed 0")


## Table 1: methods used in the one-dimensional experiment

The descriptions and reference numbers follow Table 1. The implementations are in `src/`.


In [ ]:
method_summary = pd.DataFrame([
    ("GKS", "Section SM2", "No", "—", "[40]"),
    ("S-GKS", "Section SM2", "No", "MM2", "[42, 37]"),
    ("resS-GKS", "Section SM2; Remark 3.1", "No", "MM1", "[7]"),
    ("recS-GKS", "Section SM2; Remark 3.1", "No", "MM2", "[49]"),
    ("PS-GKS", "Algorithm 4.1", "No", "MM3", "Here"),
    ("resPS-GKS", "Algorithm 4.1; Section 4.6", "No", "MM3", "Here"),
    ("recPS-GKS", "Algorithm 4.1; Section 4.6", "No", "MM3", "Here"),
    ("PS-GKB", "Algorithm SM4.1; Section SM4.1", "No", "MM3", "Here & [27]"),
    ("FGK", "Section SM4.2", "No", "MM4", "[27]"),
    ("FLSQR-I", "See [18]", "Yes", "MM5", "[18]"),
    ("FLSQR-R", "See [18]", "Yes", "MM5", "[18]"),
    ("FLSQR-W", "Remark 5.1", "Yes", "MM5", "Here"),
    ("IRW-FLSQR", "See [25]", "Yes", "MM3", "[25]"),
], columns=["Method", "Description", "Requires inverse of Psi", "MM weights", "Reference"])
method_summary.to_csv(OUT / "table_01.csv",index=False)
fig, ax = plt.subplots(figsize=(14,5.5))
ax.axis("off")
ax.set_title("Table 1. Summary of reconstruction methods",fontsize=14,pad=12)
artist = ax.table(cellText=method_summary.values,colLabels=method_summary.columns,
                  colWidths=[0.13,0.39,0.2,0.13,0.15],cellLoc="left",loc="center")
artist.auto_set_font_size(False)
artist.set_fontsize(10)
artist.scale(1,1.8)
for (row,col), entry in artist.get_celld().items():
    entry.set_edgecolor("#dddddd")
    if row==0:
        entry.set_facecolor("#e7eef5")
        entry.set_text_props(weight="bold")
    elif row%2==0:
        entry.set_facecolor("#f7f7f7")
fig.tight_layout()
fig.savefig(OUT / "table_01.png",dpi=200,bbox_inches="tight")
plt.show()

## Solver classes and parameters


In [ ]:
from msolver import VanillaGKS, MM1GKS, MM2GKS, MM3GKS, IASGKS
from mInvsolver import MM3GKSInv, IASGKSInv
from pgkbInvsolver import MM3GKBInv, IASGKBInv
from fsolver import MM4FGK, IASFGK
from f2019solver import MM5FGK2019, IASFGK2019
from fIRWsolver import MM3FGKIRW, IASFGKIRW
n_its = 150
init_dim, min_dim, max_dim = 5, 15, 25
restart_dim = 40
lambdah_min, lambdah_max = 1e-7, 1e7
fgk_breakdown_tol = 1e-10
common = dict(n_iter=n_its, regparam="dp", x_true=ground_truth,
              lambdah_min=lambdah_min, lambdah_max=lambdah_max, ksparsity=None)
gks_common = dict(init_dim=init_dim, regparam_kwargs={"eta":1.01})
args = dict(noise_var=noise_var, Lpinv_factory=Lpinv_factory, K=None)
results = {}


## Compute the initial GKS reconstruction


In [ ]:
v_solver = VanillaGKS(A, L, b, **args)
results[("V", "GKS")] = logged_solve(OUT / "comparison_1d_solver.log", "V GKS", v_solver.solve,
    x0=None, lambdah0=1.0, weight_params={}, xker=None, **common, **gks_common)
v = results[("V", "GKS")]
warm = dict(x0=v["x"], lambdah0=v["lambdah"])


## Compute MM and IAS methods

The common S-GKS calls use the previous GKS solution as a warm start. Flexible
methods start from their own default initialization. Restarted PS-GKS uses the
general implementation; full/recycled PS-GKS and
PS-GKB use its inverse-specific implementations.


In [ ]:
for weights in ["MM", "IAS"]:
    params = {"q":1.0} if weights=="MM" else {"hyperpr_r":-1.0,"hyperpr_beta":1.0}
    sgks = (MM2GKS if weights=="MM" else IASGKS)(A,L,b,**args)
    res_sgks = (MM1GKS if weights=="MM" else IASGKS)(A,L,b,**args)
    ps = (MM3GKSInv if weights=="MM" else IASGKSInv)(A,L,Linv,b,
        noise_var=noise_var,Lpinv_factory=None,K=None)
    res_ps = (MM3GKS if weights=="MM" else IASGKS)(A,L,b,**args)
    gkb = (MM3GKBInv if weights=="MM" else IASGKBInv)(A,L,Linv,b,
        noise_var=noise_var,Lpinv_factory=None,K=None)
    fgk = (MM4FGK if weights=="MM" else IASFGK)(A,L,b,**args)
    flsqr = (MM5FGK2019 if weights=="MM" else IASFGK2019)(A,L,Linv,b,noise_var=noise_var,K=None)
    irw = (MM3FGKIRW if weights=="MM" else IASFGKIRW)(A,L,Linv,b,noise_var=noise_var,K=None)
    gks_jobs = [
        ("S-GKS", sgks.solve, {}),
        ("resS-GKS", res_sgks.solve, {"restart_dim":restart_dim}),
        ("recS-GKS", sgks.solve, {"min_dim":min_dim,"max_dim":max_dim}),
        ("PS-GKS", ps.solve_with_priorconditioning, {}),
        ("resPS-GKS", res_ps.solve_with_priorconditioning, {"restart_dim":restart_dim}),
        ("recPS-GKS", ps.solve_with_priorconditioning, {"min_dim":min_dim,"max_dim":max_dim}),
    ]
    for method, solve, options in gks_jobs:
        results[(weights,method)] = logged_solve(OUT / "comparison_1d_solver.log",weights+" "+method,
            solve,weight_params=params,
            **dict(common, **({"lambdah_min":1e-6,"lambdah_max":1e6}
                             if weights=="IAS" and method=="resPS-GKS" else {})),
            **gks_common,**warm,**options)
    results[(weights,"PS-GKB")] = logged_solve(OUT / "comparison_1d_solver.log",weights+" PS-GKB",
        gkb.solve,x0=None,lambdah0=None,weight_params=params,refresh_basis=False,**common)
    flexible_jobs = [("FGK",fgk.solve,{}),
                     ("FLSQR-I",flsqr.solve,{"mode":"I"}),
                     ("FLSQR-R",flsqr.solve,{"mode":"R"}),
                     ("FLSQR-W",flsqr.solve,{"mode":"W"}),
                     ("IRW-FLSQR",irw.solve,{"mode":"IRW"})]
    for method, solve, options in flexible_jobs:
        results[(weights,method)] = logged_solve(OUT / "comparison_1d_solver.log",weights+" "+method,
            solve,x0=None,lambdah0=None,weight_params=params,breakdown_tol=fgk_breakdown_tol,
            refresh_basis=False,**common,**options)


## Table 2: final metrics and matrix-vector product counts


In [ ]:
# The IAS FLSQR-W row uses the MM FLSQR-W result.
table_results = dict(results)
table_results[("IAS","FLSQR-W")] = results[("MM","FLSQR-W")]
table = summarize(table_results)
table.to_csv(OUT / "table_02.csv",index=False)
display(table)

# Format the table for the figure.
pretty = table.copy()
for col in ["mu","kappa"]: pretty[col] = pretty[col].map(lambda value: f"{value:.0e}")
for col in ["RRE","SSIM","Gini index"]: pretty[col] = pretty[col].map(lambda value: f"{value:.3f}")
fig, ax = plt.subplots(figsize=(15,10))
ax.axis("off")
ax.set_title("Table 2. One-dimensional cosine problem",fontsize=15,pad=15)
artist = ax.table(cellText=pretty.values,colLabels=pretty.columns,
                  colWidths=[0.06,0.12,0.08,0.07,0.07,0.07,0.09,0.08,0.08,0.08,0.1],
                  cellLoc="center",loc="center")
artist.auto_set_font_size(False)
artist.set_fontsize(10)
artist.scale(1,1.75)
for (row,col), entry in artist.get_celld().items():
    entry.set_edgecolor("#dddddd")
    if row==0:
        entry.set_facecolor("#e7eef5")
        entry.set_text_props(weight="bold")
    elif row%2==0:
        entry.set_facecolor("#f7f7f7")
fig.tight_layout()
fig.savefig(OUT / "table_02.png",dpi=200,bbox_inches="tight")
plt.show()


## Figure 4: convergence of the selected methods


In [ ]:
method_colors = {"GKS":"C1", "S-GKS":"C4", "resS-GKS":"C4", "recS-GKS":"C4",
                 "PS-GKS":"C2", "resPS-GKS":"C2", "recPS-GKS":"C2", "PS-GKB":"C5",
                 "FGK":"C3", "FLSQR-I":"C3", "FLSQR-R":"C3", "FLSQR-W":"C3", "IRW-FLSQR":"C3"}
method_lines = {"GKS":"solid", "S-GKS":"solid", "resS-GKS":"dotted", "recS-GKS":"dashed",
                "PS-GKS":"solid", "resPS-GKS":"dotted", "recPS-GKS":"dashed", "PS-GKB":"dashed",
                "FGK":"dotted", "FLSQR-I":"dotted", "FLSQR-R":"dashed", "FLSQR-W":"dashdot",
                "IRW-FLSQR":(0, (3, 1, 1, 1, 1, 1))}

fig, axs = plt.subplots(2,3,figsize=(14,8))
metrics = [("rre_history","RRE history"),("ssim_history","SSIM history"),("gini_indices","Gini index")]
methods = ["GKS","S-GKS","recS-GKS","PS-GKS","recPS-GKS","PS-GKB","FGK","IRW-FLSQR","FLSQR-R","FLSQR-W"]
rows = []
for row, weights in enumerate(["MM","IAS"]):
    for col, (metric,title) in enumerate(metrics):
        ax = axs[row,col]
        for method in methods:
            values = np.asarray(results[("V" if method=="GKS" else weights,method)][metric])
            start = 5
            ax.plot(np.arange(start,len(values)),values[start:],color=method_colors[method],
                    ls=method_lines[method],lw=2,label=method)
            rows.extend({"weights":weights,"method":method,"metric":metric,"iteration_index":i,
                         "value":float(values[i])} for i in range(start,len(values)))
        ax.set(title=f"{title} ({weights} weights)",xlabel="Iteration index")
        ax.legend()
fig.tight_layout()
fig.savefig(OUT / "figure_04.png",dpi=250)
pd.DataFrame(rows).to_csv(OUT / "figure_04.csv",index=False)
plt.show()
